# ProGAN — Progressive Growing of GANs for Improved Quality, Stability, and Variation (2017)

_Papers · Generative Models_

**Grow a GAN from a 4×4 thumbnail up to a 1024×1024 photo by fading in higher-resolution layers one at a time — plus three tricks (minibatch stddev, equalized learning rate, pixel norm) that keep the training stable.**

---

This notebook is a practice scaffold for the **ProGAN — Progressive Growing of GANs for Improved Quality, Stability, and Variation (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — Equalized learning rate

ProGAN's first trick (Section 4.1) stores each weight as plain `N(0, 1)` and multiplies it by a fixed He constant `c = sqrt(2 / fan_in)` on *every* forward pass, instead of baking the scale into the initialization. Because Adam normalizes updates by the gradient's standard deviation, giving every layer the same stored-weight scale makes them all learn at the same relative speed. We build the layer and verify `c` on the paper's worked 3×3, 512-channel example.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class EqualizedConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, k, stride=1, padding=0):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_ch, in_ch, k, k))   # N(0,1) -- trivial init
        self.bias = nn.Parameter(torch.zeros(out_ch))
        fan_in = in_ch * k * k
        self.scale = (2.0 / fan_in) ** 0.5                              # c = sqrt(2 / fan_in)
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        scaled_weight = self.weight * self.scale                       # apply He scale at runtime
        return F.conv2d(x, scaled_weight, self.bias, self.stride, self.padding)

# Worked example (Part A): 3x3 conv, 512 in-channels -> fan_in = 512*9 = 4608.
layer = EqualizedConv2d(512, 256, 3, padding=1)
fan_in = 512 * 3 * 3
print("fan_in =", fan_in, "  He scale c =", round(layer.scale, 6))   # 4608, ~0.020833
assert abs(layer.scale - (2 / 4608) ** 0.5) < 1e-12
assert abs(layer.scale - 0.020833) < 1e-5      # matches the worked number 0.0208

### Step 2 — Minibatch standard deviation

The second trick (Section 3) gives the discriminator a cheap way to detect mode collapse. We compute the standard deviation of each feature at each location *across the batch*, average it down to a single scalar, then concatenate that scalar back as one constant extra channel. A diverse batch produces a large value; a collapsed batch of identical images produces ~0 — a signal the discriminator learns to punish.

In [ ]:
class MinibatchStddev(nn.Module):
    def forward(self, x):                                  # x: (N, C, H, W)
        std = x.std(dim=0, unbiased=False)                 # std over batch -> (C, H, W)
        mean_std = std.mean()                              # average to ONE scalar
        extra = mean_std.expand(x.size(0), 1, x.size(2), x.size(3))  # replicate as a map
        return torch.cat([x, extra], dim=1)                # one extra (constant) channel

mb = MinibatchStddev()
varied = torch.randn(8, 4, 4, 4)                           # a diverse batch
collapsed = torch.randn(1, 4, 4, 4).repeat(8, 1, 1, 1)     # mode collapse: 8 identical images

v_val = mb(varied)[0, -1, 0, 0].item()                     # the appended stddev channel
c_val = mb(collapsed)[0, -1, 0, 0].item()
print(f"stddev channel  varied={v_val:.3f}  collapsed={c_val:.3f}")   # varied ~1.0, collapsed ~0.0
assert v_val > 0.5 and c_val < 1e-5     # collapse -> ~0 variety, the signal D learns to punish

### Step 3 — Pixelwise feature vector normalization

The third trick (Section 4.2, generator only) rescales each pixel's feature vector to unit length: `b = a / sqrt(mean_j(a_j^2) + eps)`, normalizing across the channel dimension. This stops the generator's and discriminator's magnitudes from spiralling out of control during training. We feed in deliberately large features and confirm every pixel comes out with unit root-mean-square.

In [ ]:
class PixelNorm(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x):                                  # normalize over channel dim
        rms = torch.sqrt(x.pow(2).mean(dim=1, keepdim=True) + self.eps)
        return x / rms

pn = PixelNorm()
feat = torch.randn(2, 16, 4, 4) * 5.0                      # large magnitudes
out = pn(feat)

rms = out.pow(2).mean(dim=1).sqrt()                        # per-pixel root-mean-square
print("per-pixel RMS after PixelNorm (min/max):", round(rms.min().item(), 3), round(rms.max().item(), 3))
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-3)   # every pixel ~ unit length

### Step 4 — Progressive growing and the fade-in blend

The headline idea (Section 2): grow the generator one resolution at a time. When a new higher-resolution block is added it is *faded in* with a weight `alpha` that ramps from 0 to 1, blending `(1 - alpha)·upsample(old_rgb) + alpha·new_rgb` so the trained low-res layers are never shocked by the new block's random output. We verify the blend on a worked example, then watch a toy generator climb 4×4 → 8×8 → 16×16 as blocks (each EqualizedConv + PixelNorm + LeakyReLU) are stacked.

In [ ]:
def fade_in(old_rgb, new_rgb, alpha):
    up = F.interpolate(old_rgb, scale_factor=2, mode="nearest")   # old path: nearest upsample
    return (1 - alpha) * up + alpha * new_rgb

# Worked example (Part B): old pixel 0.20, new pixel 0.80 -> sweep alpha.
old = torch.full((1, 3, 2, 2), 0.20)
new = torch.full((1, 3, 4, 4), 0.80)
for a in (0.0, 0.5, 1.0):
    faded = fade_in(old, new, a)[0, 0, 0, 0].item()
    print(f"alpha={a}:  faded pixel = {faded:.2f}")   # 0.20, 0.50, 0.80
assert abs(fade_in(old, new, 0.5)[0, 0, 0, 0].item() - 0.50) < 1e-6

# A toy generator that GROWS 4 -> 8 -> 16, each block = EqualizedConv + PixelNorm + LeakyReLU,
# ending in a 1x1 toRGB. We just show the output resolution climbing as blocks are added.
class GenBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = EqualizedConv2d(in_ch, out_ch, 3, padding=1)
        self.norm = PixelNorm()
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode="nearest")   # double resolution
        return self.act(self.norm(self.conv(x)))

z = torch.randn(2, 32, 4, 4)                  # start at 4x4
blocks = nn.ModuleList([GenBlock(32, 32), GenBlock(32, 32)])   # -> 8x8 -> 16x16
toRGB = EqualizedConv2d(32, 3, 1)

h = z
print("start resolution:", tuple(h.shape[-2:]))
for i, b in enumerate(blocks):
    h = b(h)
    print(f"after block {i+1}: resolution {tuple(h.shape[-2:])}, RGB {tuple(toRGB(h).shape[-2:])}")
# Output: 4x4 -> 8x8 -> 16x16. This is progressive growing: each block adds one resolution.
# (Real ProGAN fades each new block in with the alpha schedule above and trains WGAN-GP --
#  see paper-wgan -- between growth steps. Our run is tiny; not the paper's numbers.)

## Visualize the data & results

_Does the fade-in weight α blend smoothly from the old resolution to the new one, and does the equalized learning rate keep training more stable than baking He init in once?_

### Step 1 — Chart the fade-in blend

First we tabulate the fade-in blend across the full `alpha` sweep from 0 to 1 for a single old/new pixel pair. This is the exact linear interpolation from the worked example, and it shows the smooth slide from the old resolution's value to the new one as `alpha` ramps up.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Left chart: the fade-in blend, exact (matches the worked example). ---
old = 0.20
new = 0.80
for i in range(11):
    a = i / 10
    faded = (1 - a) * old + a * new
    print(f"alpha={a:.1f}  faded={faded:.3f}")   # 0.20 ... 0.50 ... 0.80

### Step 2 — Equalized LR vs. He-baked-in on a toy regression

Now the ablation that justifies the equalized learning rate. We build the same small network two ways: `EqLinear` stores `N(0,1)` and scales at runtime, while `HeLinear` bakes the He scale into the initialization once. With layers of very different fan_in (4 → 256 → 8 → 1), the equalized version descends smoothly while the He-baked-in version is noisier — exactly the stability difference Section 4.1 describes.

In [ ]:
torch.manual_seed(0)

class EqLinear(nn.Module):                # equalized LR: store N(0,1), scale at runtime
    def __init__(self, i, o):
        super().__init__()
        self.w = nn.Parameter(torch.randn(o, i))
        self.b = nn.Parameter(torch.zeros(o))
        self.s = (2.0 / i) ** 0.5

    def forward(self, x):
        return F.linear(x, self.w * self.s, self.b)

class HeLinear(nn.Module):                # ablation: bake He into init, use as-is
    def __init__(self, i, o):
        super().__init__()
        self.w = nn.Parameter(torch.randn(o, i) * (2.0 / i) ** 0.5)
        self.b = nn.Parameter(torch.zeros(o))

    def forward(self, x):
        return F.linear(x, self.w, self.b)

def net(L):   # layers with very different fan_in (4 -> 256 -> 8 -> 1) to expose the effect
    return nn.Sequential(L(4, 256), nn.LeakyReLU(0.2), L(256, 8), nn.LeakyReLU(0.2), L(8, 1))

x = torch.randn(64, 4)
target = torch.full((64, 1), 2.0)

def run(L):
    torch.manual_seed(0)
    m = net(L)
    opt = torch.optim.Adam(m.parameters(), 1e-2, betas=(0.0, 0.99))
    gaps = []
    for t in range(201):
        opt.zero_grad()
        y = m(x)
        loss = ((y - target) ** 2).mean()
        loss.backward()
        opt.step()
        gaps.append(abs(y.detach().mean().item() - 2.0))   # gap of output magnitude to target
    return gaps

eq = run(EqLinear)
he = run(HeLinear)
idx = list(range(0, 201, 10))
print("equalized LR gap:", [round(eq[i], 3) for i in idx])
print("He-baked-in  gap:", [round(he[i], 3) for i in idx])
# Equalized LR descends smoothly; He-baked-in is noisier across the very-different-fan_in layers.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a toy generator trained with the equalized learning rate
            ($\mathcal{N}(0,1)$ weights, scaled by $\sqrt{2/\text{fan\_in}}$ at every forward pass). You turn
            it off &mdash; bake He init into the weights once and train normally &mdash; keeping everything
            else identical. What does the paper claim this changes, and which ProGAN idea does the ablation
            isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap only the weight handling: instead of storing $\mathcal{N}(0,1)$ and dividing by $c$ each forward pass, initialize weights directly with He init ($\mathcal{N}(0,c^2)$) and use them as-is. Keep the architecture, loss, Adam settings, and data identical. — _An honest ablation changes exactly one thing — where the per-layer scale is applied (runtime vs init) — so any difference is attributable to it._
- Train both and watch stability: with equalized LR, every layer's stored weights share one scale, so Adam moves them at the same relative speed; without it, layers with different fan_in learn at different speeds and the run is noisier / less stable. — _Per §4.1, Adam normalizes updates by the gradient's std, so identical stored-weight scale ⇒ identical learning speed across layers._
- Conclude the equalized learning rate (§4.1) is doing real work — it is not just a fancy way to write He init. — _Mathematically one forward pass is identical to He init; the benefit only appears through the optimizer over many steps, which the ablation exposes._

**Answer:** Turning equalized LR off makes training less stable / more uneven across layers. Both
                 setups are identical in a single forward pass (storing $\mathcal{N}(0,1)$ and dividing by $c$ is
                 numerically the same as He init), so the difference appears only through the optimizer:
                 with equalized LR, Adam sees the same weight scale in every layer and moves them at the same
                 relative speed (§4.1, "the dynamic range, and thus the learning speed, is the same for all
                 weights"). The ablation isolates the equalized learning rate as the cause. The CODEVIZ
                 panel shows this contrast on our toy run.

</details>

**Problem 2.** Compute the equalized-LR constant $c$ for a generator's $1\times1$ toRGB convolution
            that maps $256$ feature channels to $3$ RGB channels. Then state the scaled weight $\hat{w}$
            for a stored weight $w=1$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find fan_in: for a $1\times1$ conv, $\text{fan\_in} = (\text{in\_channels})\times k_h\times k_w = 256\times1\times1 = 256$. — _Kernel is $1\times1$, so each output reads exactly the 256 input channels at that pixel._
- Compute $c=\sqrt{2/256}=\sqrt{0.0078125}\approx 0.08839$. — _This is the He scale; ProGAN stores $\mathcal{N}(0,1)$ weights and applies this scale at runtime._
- Read the §4.1 formula literally: $\hat{w}=w/c$. (In practice ProGAN multiplies the runtime weight by the He std scale so the effective weight std is $c$; the printed/checked quantity here is $c\approx0.0884$ itself.) — _We report the unambiguous quantity $c$, which the notebook verifies equals $(2/256)^{0.5}$._

**Answer:** $\text{fan\_in}=256\times1\times1=256$, so $c=\sqrt{2/256}=\sqrt{0.0078125}\approx
                 \mathbf{0.0884}$. A $1\times1$ conv has a much smaller fan_in than a $3\times3$, so its He
                 scale ($\approx0.0884$) is about $4.24\times$ larger than the $3\times3$/512-in layer's
                 ($\approx0.0208$) — exactly why per-layer scaling matters. The notebook checks
                 $c=(2/256)^{0.5}$.

</details>

**Problem 3.** During a 8&times;8 &rarr; 16&times;16 transition, the upsampled old path gives a pixel value $0.30$
            and the new 16&times;16 block gives $0.90$. What is the faded output at $\alpha=0$, $\alpha=0.25$,
            and $\alpha=1$? What would happen if you instead jumped straight to $\alpha=1$ on step one?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Apply the blend $(1-\alpha)\cdot0.30 + \alpha\cdot0.90$. At $\alpha=0$: $1\cdot0.30+0=0.30$. — _At the start of a transition you trust only the already-trained low-res path._
- At $\alpha=0.25$: $0.75\cdot0.30 + 0.25\cdot0.90 = 0.225 + 0.225 = 0.45$. — _A quarter of the way in, the new block contributes a quarter of the output._
- At $\alpha=1$: $0\cdot0.30 + 1\cdot0.90 = 0.90$. Jumping to $\alpha=1$ immediately feeds the new block's random, untrained output straight into the network. — _The new block starts with random weights; full weight on it before it has learned shocks the trained low-res layers (§2)._

**Answer:** Outputs: $\alpha=0\Rightarrow0.30$, $\alpha=0.25\Rightarrow0.45$, $\alpha=1\Rightarrow0.90$
                 — a smooth slide from old to new. Jumping straight to $\alpha=1$ on step one would feed the new
                 block's random output at full weight into the well-trained low-res layers, causing the
                 "sudden shock" the fade-in exists to prevent (§2). That is why $\alpha$ ramps linearly $0\to1$.

</details>